# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Accessing dataset metadata (no subscripting)
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their IDs. All Croissant dataset entities are referenced using their `@id`.

Let's print details about available record sets (`cr:RecordSet`) and their fields and columns.

In [ ]:
# List all record set @ids
print("Record sets available in this dataset:")
for rs in dataset.metadata.record_sets:
    print(f"@id: {rs.id}, name: {rs.name}")

# Optionally, show the fields for each record set
for rs in dataset.metadata.record_sets:
    print(f"\nFields in RecordSet with @id={rs.id}:")
    for field in rs.fields:
        print(f"  Field @id: {field.id}, name: {field.name}, dataType: {getattr(field, 'data_type', None)}")
    print("Columns:")
    for column in rs.columns:
        print(f"  Column @id: {column.id}, name: {column.name}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

The first available record set will be extracted as an example. Use the printed `@id` values above to adjust as needed.

In [ ]:
# Extract all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

print(f"All RecordSet @ids: {record_set_ids}")
# For demonstration, select the first one
selected_record_set_id = record_set_ids[0] if len(record_set_ids) > 0 else None
if selected_record_set_id is not None:
    print(f"\nLoading records from RecordSet: {selected_record_set_id}\n")
    records = list(dataset.records(record_set=selected_record_set_id))
    df = pd.DataFrame(records)
    print("DataFrame columns:", df.columns.tolist())
    display(df.head())
else:
    print("No record sets found in the metadata.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, and grouping data by key attributes for further analysis.

All field names/columns referenced below use their `@id` as listed in the overview.

In [ ]:
import numpy as np

# Check DataFrame is loaded
if selected_record_set_id is not None and not df.empty:
    # List numeric fields/columns by data type (heuristic)
    print("Numeric columns detected:")
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    print(numeric_columns)
    # Pick one numeric field @id for demonstration
    numeric_field_id = numeric_columns[0] if len(numeric_columns) > 0 else None
    
    if numeric_field_id is not None:
        threshold = df[numeric_field_id].mean()  # Just as an example
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by a categorical column (non-numeric)
        group_fields = [col for col in df.columns if col not in numeric_columns]
        group_field_id = group_fields[0] if group_fields else None
        if group_field_id is not None:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
        else:
            print("No categorical field (non-numeric) available for grouping.")
    else:
        print("No numeric field detected for this record set.")
else:
    print("No data loaded for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset, using matplotlib and seaborn.

We plot the distribution of the first numeric field and a boxplot grouped by the first categorical field (`@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if selected_record_set_id is not None and not df.empty and numeric_field_id is not None:
    # Histogram
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot grouped by first categorical field, if available
    if group_field_id is not None:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available for plot.")

## 6. Conclusion
In this notebook, we explored the Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells dataset using the `mlcroissant` library. We loaded metadata, reviewed the structure (record sets and field `@id`s), extracted records as Pandas DataFrames, performed basic EDA (filtering, normalization, and grouping), and visualized distributions.

For deeper analysis, refer to the field and record set `@id`s revealed to further filter, join, or transform this dataset for your research questions. The `mlcroissant` API provides structured access to entity metadata for principled, reproducible data science with FAIR datasets.